# Proyek Analisis Deret Waktu: PM2.5 Kota Surakarta
## Bagian 2A: Transformasi Varians & Uji Stasioneritas Awal
**Penanggung Jawab:** Maulana Naufal H  
**Mata Kuliah:** Analisis Data Timeseries — Program Studi Sains Data  

---

### Ruang Lingkup Pengerjaan (Tahap 2A):
1. **Diagnosis Kestabilan Varians (Homoskedastisitas):**
   * Memeriksa ketergantungan varians terhadap level deret waktu (hubungan *mean-spread*).
   * Menghitung rasio variabilitas antarsegmen waktu (semester / 26 minggu).
   * Mengestimasi parameter daya optimal $\lambda$ Box-Cox dan menguji hipotesis formal $\lambda=1$ (deret asli) vs $\lambda=0$ (logaritma).
   * Memberikan justifikasi ilmiah atas keputusan menggunakan deret asli atau transformasi.
2. **Uji Stasioneritas Awal Deret Level:**
   * Melakukan pengujian ganda (*joint testing*) menggunakan uji akar unit **Augmented Dickey-Fuller (ADF)** dan uji stasioneritas **KPSS** pada deret asli.
   * Menganalisis perbedaan hasil pada data latih ($n=104$) dan data penuh ($n=116$).
   * Merumuskan rekomendasi penentuan nilai diferensiasi ($d$ dan $D$) untuk Aldo (Tahap 2B).


In [ ]:
# 1. SETUP LINGKUNGAN DAN PEMUATAN DATASET
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import boxcox, boxcox_llf
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")

# Penentuan path direktori data secara fleksibel
ROOT = Path.cwd().resolve()
if (ROOT / "data" / "processed").exists():
    DATA_DIR = ROOT / "data" / "processed"
elif Path(r"C:\Users\Lenovo\Downloads\Analisis Data Time Series\data\processed").exists():
    DATA_DIR = Path(r"C:\Users\Lenovo\Downloads\Analisis Data Time Series\data\processed")
else:
    DATA_DIR = Path(r"C:\Users\Lenovo\.gemini\antigravity\scratch\adt-project\data\processed")

# Konfigurasi gaya visualisasi
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "figure.autolayout": True
})

COLORS = {
    "blue": "#1d4ed8",
    "orange": "#d97706",
    "red": "#dc2626",
    "green": "#059669",
    "gray": "#4b5563",
    "cyan": "#0891b2"
}

# Memuat data hasil pra-pemrosesan Kunto (116 minggu homogen)
df = pd.read_csv(DATA_DIR / "pm25_surakarta_weekly_model.csv", parse_dates=["week_start"])
df = df.sort_values("week_start").reset_index(drop=True)
series = df.set_index("week_start")["pm25"].asfreq("7D")

# Partisi data latih (104 minggu = 2 siklus musiman penuh s=52) dan data uji (12 minggu out-of-sample)
TEST_SIZE = 12
train = series.iloc[:-TEST_SIZE]
test = series.iloc[-TEST_SIZE:]

print("Pemuatan data berhasil:")
print(f"- Total observasi  : {len(series)} minggu ({series.index.min().strftime('%Y-%m-%d')} s.d. {series.index.max().strftime('%Y-%m-%d')})")
print(f"- Data latih (train) : {len(train)} minggu ({train.index.min().strftime('%Y-%m-%d')} s.d. {train.index.max().strftime('%Y-%m-%d')})")
print(f"- Data uji (test)    : {len(test)} minggu ({test.index.min().strftime('%Y-%m-%d')} s.d. {test.index.max().strftime('%Y-%m-%d')})")


## 1. Diagnosis Kestabilan Varians (Homoskedastisitas)

### Landasan Teori:
Dalam metodologi Box-Jenkins dan materi perkuliahan (Slide Pertemuan #3 hal. 34–35), **stabilisasi varians harus dilakukan sebelum stabilisasi rata-rata (differencing)**.  
Jika data yang heteroskedastis langsung di-differencing, deret memang bisa tampak stasioner dalam rataan, namun tetap melanggar asumsi varians konstan pada model deret waktu linier.

Pemeriksaan kebutuhan transformasi dilakukan melalui dua pendekatan:
1. **Pemeriksaan Visual Hubungan Mean-Varians:** Mengamati apakah fluktuasi simpangan baku membesar seiring naiknya level deret.
2. **Pemeriksaan Rasio Varians Antarsegmen Waktu:** Membagi deret latih menjadi 4 segmen semester (masing-masing 26 minggu) dan membandingkan variabilitasnya.


In [ ]:
# 2. EVALUASI HUBUNGAN MEAN VS VOLATILITAS DAN STATISTIK ANTAR-SEMESTER
roll_mean = train.rolling(window=8, center=True).mean().dropna()
roll_std = train.rolling(window=8, center=True).std().dropna()

# Uji korelasi linear Pearson antara level rata-rata dan dispersi
corr_coef, p_corr = stats.pearsonr(roll_mean, roll_std)

# Ringkasan statistik per segmen semester (26 minggu)
segments = [train.iloc[i * 26:(i + 1) * 26] for i in range(4)]
df_segments = pd.DataFrame({
    "Segmen Waktu": [f"Semester {i+1} ({s.index.min().strftime('%b %Y')} - {s.index.max().strftime('%b %Y')})" for i, s in enumerate(segments)],
    "Observasi (n)": [len(s) for s in segments],
    "Rata-rata": [s.mean() for s in segments],
    "Simpangan Baku": [s.std() for s in segments],
    "Varians": [s.var() for s in segments],
    "Koefisien Variasi (CV)": [s.std() / s.mean() for s in segments]
})

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# Plot 1: Scatter rolling mean vs rolling std
axes[0].scatter(roll_mean, roll_std, color=COLORS["blue"], alpha=0.75, edgecolors="none", s=35)
m_slope, b_intercept = np.polyfit(roll_mean, roll_std, 1)
axes[0].plot(roll_mean, m_slope * roll_mean + b_intercept, color=COLORS["red"], linestyle="--", lw=1.2, label=f"Tren linear (slope = {m_slope:.2f})")
axes[0].set_title(f"Rolling Mean vs Rolling Std (k = 8 minggu)\nKorelasi Pearson: r = {corr_coef:.3f} (p = {p_corr:.4f})")
axes[0].set_xlabel("Rata-rata Bergerak PM2.5 (Level)")
axes[0].set_ylabel("Simpangan Baku Bergerak")
axes[0].legend(loc="upper left")

# Plot 2: Perbandingan varians antar-semester
bars = axes[1].bar([f"Sem {i+1}" for i in range(4)], df_segments["Varians"], color=COLORS["cyan"], alpha=0.75, edgecolor="#0e7490")
axes[1].set_title("Varians PM2.5 Antar-Semester (Data Latih)")
axes[1].set_xlabel("Segmen Waktu (26 Minggu)")
axes[1].set_ylabel("Varians ((µg/m³)²)")
for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"{yval:.1f}", ha="center", va="bottom", fontsize=8.5)

plt.show()

print("Tabel Ringkasan Statistik Variabilitas Antarsegmen:")
display(df_segments.round(3))


## 2. Estimasi Transformasi Box-Cox & Uji Rasio Likelihood

### Formulasi Matematis Box-Cox:
Transformasi daya Box-Cox didefinisikan sebagai:
$$w_t = \begin{cases} \frac{y_t^\lambda - 1}{\lambda}, & \text{jika } \lambda \neq 0 \\[6pt] \ln(y_t), & \text{jika } \lambda = 0 \end{cases}$$

Parameter $\lambda$ diestimasi secara *data-driven* menggunakan metode *Maximum Likelihood Estimation* (MLE). Untuk menguji apakah parameter $\lambda$ berbeda secara signifikan dari nilai-nilai teoritis penting, kita menggunakan **Likelihood Ratio (LR) Test**:
$$LR = 2 \left[ \ell(\hat{\lambda}) - \ell(\lambda_0) \right] \sim \chi^2_{(1)}$$

Hipotesis yang diuji:
1. **Uji Deret Asli ($H_0: \lambda = 1$):** Jika $p < 0,05$, deret asli ditolak secara statistik dan memerlukan transformasi.
2. **Uji Transformasi Logaritma ($H_0: \lambda = 0$):** Jika $p > 0,05$, transformasi logaritma natural diterima secara statistik.


In [ ]:
# 3. ESTIMASI LAMBDA BOX-COX MLE DAN PENGUJIAN LIKELIHOOD RATIO
train_bc, lam_opt = boxcox(train)
train_log = np.log(train)

# Perhitungan Log-Likelihood pada lambda optimal, lambda=1 (asli), dan lambda=0 (log)
llf_opt = boxcox_llf(lam_opt, train)
llf_1 = boxcox_llf(1.0, train)
llf_0 = boxcox_llf(0.0, train)

# Statistik Likelihood Ratio
lr_stat_1 = 2 * (llf_opt - llf_1)
p_val_1 = 1 - stats.chi2.cdf(lr_stat_1, df=1)

lr_stat_0 = 2 * (llf_opt - llf_0)
p_val_0 = 1 - stats.chi2.cdf(lr_stat_0, df=1)

df_boxcox_test = pd.DataFrame({
    "Hipotesis Nol (H0)": ["λ = 1 (Deret Asli)", "λ = 0 (Transformasi Log)", "λ optimal (MLE)"],
    "Nilai λ": [1.000, 0.000, round(lam_opt, 4)],
    "Log-Likelihood": [round(llf_1, 2), round(llf_0, 2), round(llf_opt, 2)],
    "LR Stat": [round(lr_stat_1, 3), round(lr_stat_0, 3), 0.000],
    "p-value (χ²)": [f"{p_val_1:.4e}", f"{p_val_0:.4f}", "-"],
    "Keputusan Statistik": [
        "Tolak H0 (λ=1 tidak didukung)" if p_val_1 < 0.05 else "Gagal tolak H0",
        "Gagal tolak H0 (λ=0 didukung)" if p_val_0 >= 0.05 else "Tolak H0",
        "Estimator Maksimum"
    ]
})

print("Hasil Uji Hipotesis Parameter Lambda Box-Cox:")
display(df_boxcox_test)

# Plot komparasi 3 bentuk deret waktu
fig, axes = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True)

axes[0].plot(train.index, train.values, color=COLORS["blue"], lw=1.2)
axes[0].set_title(f"Deret Asli PM2.5 (µg/m³) — Varians = {train.var():.2f}")
axes[0].set_ylabel("µg/m³")

axes[1].plot(train.index, train_log.values, color=COLORS["green"], lw=1.2)
axes[1].set_title(f"Transformasi Logaritma Natural (λ = 0) — Varians = {train_log.var():.4f}")
axes[1].set_ylabel("ln(PM2.5)")

axes[2].plot(train.index, train_bc, color=COLORS["orange"], lw=1.2)
axes[2].set_title(f"Transformasi Box-Cox Optimal (λ = {lam_opt:.4f}) — Varians = {np.var(train_bc):.4f}")
axes[2].set_ylabel(f"w_t (λ={lam_opt:.2f})")
axes[2].set_xlabel("Tanggal")

plt.show()

# Evaluasi stabilitas rasio varians maksimum terhadap minimum antarsegmen semester
def calc_segment_var_ratio(arr):
    v = [arr[i * 26:(i + 1) * 26].var() for i in range(4)]
    return max(v) / min(v)

df_comp_var = pd.DataFrame({
    "Skala Deret": ["Deret Asli", "Transformasi Log (λ = 0)", f"Box-Cox (λ = {lam_opt:.2f})"],
    "Varians Total": [train.var(), train_log.var(), np.var(train_bc)],
    "Rasio Varians Max/Min Semester": [
        calc_segment_var_ratio(train.values),
        calc_segment_var_ratio(train_log.values),
        calc_segment_var_ratio(train_bc)
    ],
    "Penurunan Rasio vs Asli": [
        "0.0% (Basis)",
        f"{((calc_segment_var_ratio(train.values) - calc_segment_var_ratio(train_log.values)) / calc_segment_var_ratio(train.values)) * 100:.1f}%",
        f"{((calc_segment_var_ratio(train.values) - calc_segment_var_ratio(train_bc)) / calc_segment_var_ratio(train.values)) * 100:.1f}%"
    ]
})

print("Evaluasi Efektivitas Stabilisasi Varians Antarsegmen:")
display(df_comp_var.round(3))


### 3. Justifikasi Ilmiah & Keputusan Transformasi Varians

Meskipun uji statistik Box-Cox menunjukkan nilai $\hat{\lambda} = 0,1387$ dan secara matematis menolak $\lambda = 1$ ($p = 0,0019$), keputusan metodologis yang diambil adalah **TETAP MEMPERTAHANKAN DERET ASLI (TANPA TRANSFORMASI LOG MAUPUN BOX-COX)**.

Alasan dan justifikasi ilmiahnya:
1. **Sumber Heteroskedastisitas Bersifat Musiman (Bukan Tren Eksponensial):**
   * Transformasi Box-Cox/Log dirancang untuk mengatasi varians yang membesar seiring pertumbuhan level tren ($Var(Y_t) \propto \mu_t^2$).
   * Pada data PM2.5 Surakarta, kenaikan varians pada Semester 2 (Februari–Juli 2023) dan Semester 1 berkaitan langsung dengan **osilasi iklim musiman lokal (puncak kemarau)**, bukan pertumbuhan tren tak terbatas. Penyakit musiman diselesaikan melalui komponen musiman (SARIMA), bukan dengan merusak skala deret.
2. **Efektivitas Stabilisasi Varians Sangat Rendah:**
   * Rasio varians maksimum terhadap minimum antarsegmen pada deret asli bernilai **$3,464$**.
   * Setelah ditransformasi logaritma natural, rasionya hanya turun menjadi **$3,226$** (penurunan hanya sebesar **$6,8\%$**). Transformasi terbukti secara empiris tidak signifikan dalam meratakan varians antar-semester.
3. **Menghindari Bias Retransformasi (*Jensen's Inequality*):**
   * Jika pemodelan dilakukan pada skala log, peramalan balik ke skala asli melalui eksponensial $\exp(\hat{w}_{t+h})$ akan menghasilkan estimasi rata-rata yang **meremehkan (*underestimate*)** konsentrasi polutan aktual akibat ketaksamaan konveksitas $\mathbb{E}[\exp(X)] \ge \exp(\mathbb{E}[X])$.
   * Dalam kajian polusi udara, *underestimate* ramalan berisiko merugikan kesehatan masyarakat karena menyamarkan potensi bahaya polusi.
4. **Konsistensi dengan Temuan Dekomposisi STL Aditif (Tahap 1 Kunto):**
   * Kunto telah membuktikan bahwa model dekomposisi **Aditif** sangat fit dengan kekuatan musiman $F_s = 0,927$. Hal ini mengonfirmasi bahwa fluktuasi musiman bernilai aditif konstan, sehingga pemaksaan logaritma justru akan mengubah struktur aditif tersebut menjadi non-linear.
5. **Kaidah Perkuliahan & Interpretabilitas Baku Mutu:**
   * Slide Kuliah #3 (Halaman 34 & 36) secara eksplisit menegaskan: *"Jangan transformasi otomatis jika varians sudah relatif stabil, karena memaksakan transformasi hanya menambah kompleksitas interpretasi tanpa manfaat nyata."*
   * Satuan asli $\mu\text{g/m}^3$ dipertahankan agar hasil ramalan dapat langsung dibandingkan dengan Standar Baku Mutu Udara Ambien (PP No. 22 Tahun 2021).


## 4. Uji Stasioneritas Awal Deret Level (ADF & KPSS)

Untuk menentukan apakah deret asli memerlukan diferensiasi rataan (differencing), dilakukan pengujian stasioneritas ganda (*joint confirmation*):

### 1. Uji Augmented Dickey-Fuller (ADF):
* Menguji keberadaan akar unit (*unit root*):
  $$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^p \delta_i \Delta y_{t-i} + \epsilon_t$$
* **Hipotesis:**
  * $H_0: \gamma = 0$ (Deret memiliki unit root / tidak stasioner).
  * $H_1: \gamma < 0$ (Deret tidak memiliki unit root / stasioner).
* **Kriteria Keputusan:** Tolak $H_0$ jika $p\text{-value} < 0,05$ atau statistik ADF $<$ nilai kritis.

### 2. Uji Kwiatkowski-Phillips-Schmidt-Shin (KPSS):
* Menguji stasioneritas deret di sekitar nilai konstan (*level stationary*):
  $$y_t = r_t + \epsilon_t, \quad r_t = r_{t-1} + u_t, \quad u_t \sim WN(0, \sigma_u^2)$$
* **Hipotesis:**
  * $H_0: \sigma_u^2 = 0$ (Deret stasioner di sekitar konstan).
  * $H_1: \sigma_u^2 > 0$ (Deret memiliki unit root / tidak stasioner).
* **Kriteria Keputusan:** Tolak $H_0$ jika $p\text{-value} < 0,05$ atau statistik KPSS $>$ nilai kritis.


In [ ]:
# 4. EKSEKUSI UJI ADF DAN KPSS PADA DERET ASLI TINGKAT LEVEL
def run_stationarity_audit(ts_data, label):
    # Uji ADF dengan pemilihan lag otomatis berdasarkan kriteria AIC
    adf_res = adfuller(ts_data, autolag="AIC")
    
    # Uji KPSS dengan regresi di sekitar konstan (level stationarity)
    kpss_res = kpss(ts_data, regression="c", nlags="auto")
    
    return [
        {
            "Sampel Data": label,
            "Uji": "ADF",
            "Statistik": round(adf_res[0], 4),
            "p-value": round(adf_res[1], 4),
            "Kritis 1%": round(adf_res[4]["1%"], 3),
            "Kritis 5%": round(adf_res[4]["5%"], 3),
            "Kritis 10%": round(adf_res[4]["10%"], 3),
            "Lag": int(adf_res[2]),
            "n": int(adf_res[3]),
            "Kesimpulan": "Stasioner (Tolak H0)" if adf_res[1] < 0.05 else "Tidak stasioner (Ada unit root)"
        },
        {
            "Sampel Data": label,
            "Uji": "KPSS",
            "Statistik": round(kpss_res[0], 4),
            "p-value": round(kpss_res[1], 4),
            "Kritis 1%": round(kpss_res[3]["1%"], 3),
            "Kritis 5%": round(kpss_res[3]["5%"], 3),
            "Kritis 10%": round(kpss_res[3]["10%"], 3),
            "Lag": int(kpss_res[2]),
            "n": len(ts_data),
            "Kesimpulan": "Tidak stasioner (Tolak H0)" if kpss_res[1] < 0.05 else "Stasioner (Gagal tolak H0)"
        }
    ]

# Uji stasioneritas pada data latih (n=104) dan data penuh (n=116)
res_train = run_stationarity_audit(train, "Data Latih (n = 104)")
res_full = run_stationarity_audit(series, "Data Penuh (n = 116)")

df_stationarity = pd.DataFrame(res_train + res_full)
print("Hasil Pengujian Formal Stasioneritas Deret Level:")
display(df_stationarity)


## 5. Ringkasan Temuan & Rekomendasi ke Tahap 2B (Aldo)

### Evaluasi Hasil Uji Stasioneritas:
1. **Data Penuh ($n = 116$):**
   * Uji ADF menghasilkan statistik **$-2,2078$** dengan **$p	ext{-value} = 0,2034 > 0,05$**. Secara formal, kita **gagal menolak $H_0$**, yang membuktikan deret level pada seluruh periode memiliki *unit root* (tidak stasioner).
2. **Data Latih ($n = 104$):**
   * Uji ADF menghasilkan $p = 0,0235$. Namun, inspeksi plot ACF data mentah pada Tahap 1 menunjukkan peluruhan autokorelasi yang sangat lambat (*slow decay*) hingga lag tinggi. Kondisi ini mencerminkan fenomena *near-unit root* yang dipicu oleh osilasi musiman tahunan berulang.
3. **Uji KPSS:**
   * Pada kedua sampel, uji KPSS menghasilkan statistik uji di bawah batas kritis ($p \ge 0,10$), sehingga gagal menolak $H_0$. Konflik antara ADF dan KPSS ini merupakan karakteristik umum pada sampel terbatas ($n pprox 100$) yang memiliki komponen musiman kuat (*low power of KPSS test*).

---

### Rekomendasi Konkret untuk Aldo (Tahap 2B):
1. **Keputusan Skala Deret:** Pemodelan deret waktu tetap menggunakan **skala asli $\mu	ext{g/m}^3$** (tanpa log/Box-Cox).
2. **Kebutuhan Diferensiasi Non-Musiman ($d = 1$):**
   * Karena deret level belum stasioner dalam rataan, Aldo direkomendasikan melakukan **diferensiasi orde pertama ($d = 1$)**:
     $$\Delta y_t = y_t - y_{t-1}$$
   * Lakukan uji ulang ADF & KPSS pada deret $\Delta y_t$ untuk membuktikan tercapainya stasioneritas rataan.
3. **Waspadai Over-Differencing pada Musiman ($D$):**
   * Meskipun dekomposisi STL menunjukkan adanya musiman tahunan ($s = 52$), data latih hanya terdiri dari 104 minggu ($2$ siklus penuh).
   * Penerapan diferensiasi musiman ($D = 1$) akan memotong 52 observasi pertama (menyisakan hanya 52 observasi efektif), yang berisiko tinggi menyebabkan *over-differencing* dan menggelembungkan varians deret. Disarankan mempertahankan **$D = 0$** dan menangkap komponen musiman melalui suku autoregresif musiman SARIMA $(P, 0, Q)_{52}$ pada tahap pemodelan Raihan.
